# Imports

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import brentq
from scipy.special import j0, j1

PODATKI = Path("../11_vaje/navodila")
if not PODATKI.exists():
    PODATKI = Path("../../11_vaje/navodila")

# Funkcije

In [ ]:
# --- Naloga 1: palica (1D, FTCS) ---
def ftcs_1d(T, r, n_korakov):
    """n_korakov eksplicitne sheme FTCS; robni točki imata stalno temperaturo. r = D dt / dx^2."""
    T = T.copy()
    for _ in range(n_korakov):
        # TODO: T_i <- T_i + r (T_{i+1} - 2 T_i + T_{i-1}) za notranje točke (vektorsko, z rezinami)
        pass
    return T


# --- Naloga 2: tla ---
def tla(T_pov, z, D, dt, T0, leta=2):
    """Temperatura v tleh pod površjem s temperaturo T_pov (urne vrednosti, linearno interpolirane).
    Spodnji rob (z[-1]) je adiabaten. Leto se ponovi leta-krat (prva leta za ustalitev).
    Vrne urne profile zadnjega leta, oblika (len(T_pov), len(z))."""
    r = D * dt / (z[1] - z[0])**2
    kor = int(round(3600 / dt))                      # korakov v eni uri
    T = np.full(len(z), float(T0))
    zap = np.empty((len(T_pov), len(z)))
    for _ in range(leta):
        for k in range(len(T_pov)):
            Ta, Tb = T_pov[k], T_pov[(k + 1) % len(T_pov)]
            for s in range(kor):
                T[0] = Ta + (Tb - Ta) * s / kor      # robni pogoj na površju
                # TODO: korak FTCS za notranje točke; na dnu zrcalni sosed T[-2] (dT/dz = 0)
                pass
            zap[k] = T
    return zap


def harmonik(y, t, perioda):
    """Prilagodi y ~ a + b cos(w t) + c sin(w t) z w = 2 pi / perioda (linearni najmanjši kvadrati).
    Vrne amplitudo in fazo (faza narašča z zamikom)."""
    w = 2 * np.pi / perioda
    # TODO: matrika [1, cos, sin], np.linalg.lstsq; amplituda = hypot(b, c), faza = arctan2(c, b)
    pass


# --- Naloga 3: plošča, valj, krogla ---
def radialno(T0, T_ok, d, D, R, N, t_kon, r=0.15):
    """Ohlajanje/segrevanje telesa s polmerom R (d = 0 plošča s polovično debelino R, 1 valj, 2 krogla).
    Začetna temperatura T0, površje ima ves čas T_ok. Mreža r_i = i dr, i = 0..N.
    Vrne (časi, temperatura v središču, končni profil)."""
    dr = R / N
    dt = r * dr**2 / D
    n = int(np.ceil(t_kon / dt))
    T = np.full(N + 1, float(T0))
    T[-1] = T_ok
    i = np.arange(1, N)
    Tc = np.empty(n + 1)
    Tc[0] = T[0]
    for k in range(n):
        Tn = T.copy()
        # TODO: notranje točke z dodatnim členom d/(2i) (T_{i+1} - T_{i-1}); središče: (1 + d) * 2 (T_1 - T_0)
        pass
        T = Tn
        Tc[k + 1] = T[0]
    return np.arange(n + 1) * dt, Tc, T


# --- Naloga 4: vogal stene (2D, stacionarno) ---
ZID, NOTRI, ZUNAJ, REZ = 0, 1, 2, 3


def vogal(a, d, h):
    """Prerez zunanjega vogala stene debeline d na območju a x a s celicami velikosti h.
    Vrne tabelo tipov celic z obrobo (n+2) x (n+2): zunaj pri x < 0 ali y < 0, rez (adiabatno) pri x, y > a."""
    n = int(round(a / h))
    xc = (np.arange(n) + 0.5) * h
    X, Y = np.meshgrid(xc, xc)
    tip = np.full((n + 2, n + 2), REZ)
    tip[1:-1, 1:-1] = np.where((X < d) | (Y < d), ZID, NOTRI)
    tip[0, :] = ZUNAJ
    tip[:, 0] = ZUNAJ
    return tip


def prevodnosti(tip, h, lam, h_i, h_e):
    """Prevodnosti [W/(m K)] med celico zidu in njenimi štirimi sosedi (gor, dol, levo, desno).
    Zid-zid: lam; zid-zrak: h / (h / (2 lam) + 1 / h_zrak); rez: 0."""
    G_zrak = {NOTRI: h / (h / (2 * lam) + 1 / h_i), ZUNAJ: h / (h / (2 * lam) + 1 / h_e)}
    sredina = tip[1:-1, 1:-1] == ZID
    G = []
    for s in (tip[:-2, 1:-1], tip[2:, 1:-1], tip[1:-1, :-2], tip[1:-1, 2:]):
        g = np.select([s == ZID, s == NOTRI, s == ZUNAJ], [lam, G_zrak[NOTRI], G_zrak[ZUNAJ]], 0.0)
        G.append(np.where(sredina, g, 0.0))
    return G


def jacobi(tip, G, T_i, T_e, T_zac, tol=1e-7, max_iter=100000):
    """Stacionarna temperatura zidu z Jacobijevo iteracijo: T celice = utežno povprečje sosedov (uteži G).
    Celice zraka imajo stalno temperaturo (T_i notri, T_e zunaj). Vrne (T z obrobo, število iteracij)."""
    zid = tip[1:-1, 1:-1] == ZID
    T = np.where(tip == NOTRI, T_i, np.where(tip == ZUNAJ, T_e, 0.0))
    T[1:-1, 1:-1][zid] = T_zac[zid]
    vsota_G = sum(G)
    for it in range(max_iter):
        # TODO: vsota G_k * T_soseda_k po štirih sosedih (rezine kot v prevodnosti), deli z vsota_G; ustavi, ko je največja sprememba < tol
        pass
    return T, it + 1


def tok_notri(T, tip, G, T_i):
    """Toplotni tok [W/m] iz notranjega zraka v vsako celico zidu (n x n)."""
    q = np.zeros_like(G[0])
    for g, s in zip(G, (tip[:-2, 1:-1], tip[2:, 1:-1], tip[1:-1, :-2], tip[1:-1, 2:])):
        q += np.where(s == NOTRI, g * (T_i - T[1:-1, 1:-1]), 0.0)
    return q

# Testno območje

In [ ]:
# P1: največji časovni korak za tla
D_tla = 5e-7
print(f"P1: dt_max (dx = 1 cm) = {0.5 * 0.01**2 / D_tla:.0f} s")
# P2: globina prodiranja
for ime, P, enota, s in (("dan", 86400, "h", 3600), ("leto", 365 * 86400, "dni", 86400)):
    delta = np.sqrt(2 * D_tla / (2 * np.pi / P))
    print(f"P2: {ime}: delta = {delta:.3f} m, zamik na globini delta = {P / (2 * np.pi) / s:.2f} {enota}")
# P3: lastne vrednosti in časovne konstante (enote R^2/D)
beta = {0: np.pi / 2, 1: brentq(j0, 2, 3), 2: np.pi}
for d, ime in ((0, "plošča"), (1, "valj"), (2, "krogla")):
    print(f"P3: {ime:7s} beta^2 = {beta[d]**2:.3f}, tau = {1 / beta[d]**2:.4f} R^2/D")
# P4: jajce
R_j, D_j = 0.022, 1.4e-7
print(f"P4: t(70 °C) = {R_j**2 / (np.pi**2 * D_j) * np.log(2 / (30 / 95)) / 60:.1f} min")

# Naloga 1 - Slika 2

In [ ]:
L, D_pal, N_pal = 1.0, 1e-4, 51
x = np.linspace(0, L, N_pal)
dx = x[1] - x[0]
tau_pal = L**2 / (np.pi**2 * D_pal)
T_zac = 100 * np.sin(np.pi * x / L)
fig = plt.figure(figsize=(12, 4))
ax1, ax2 = fig.add_subplot(121), fig.add_subplot(122)
# TODO: za r = 0.25, 0.45, 0.50, 0.55 izračunaj T po t = 1000 s, primerjaj z analitično rešitvijo (Preglednica 1)
pass
ax1.set_title("stabilno", loc="left")
ax2.set_title("meja in nestabilnost", loc="left")
for ax in (ax1, ax2):
    ax.set_xlabel("$x$ [m]")
    ax.set_ylabel("$T$ [°C]")
    ax.legend()
fig.tight_layout()
plt.show()

# Naloga 2 - Slika 3

In [ ]:
vreme = pd.read_csv(PODATKI / "Vreme_urni.csv")
T_zrak = vreme["T"].to_numpy()
dz, H = 0.02, 8.0
z = np.arange(0, H + dz / 2, dz)
dt_tla = 300.0
print(f"r = {D_tla * dt_tla / dz**2:.3f}, povprečna temperatura zraka = {T_zrak.mean():.2f} °C")
# TODO: simulacija tal za dve leti (začetna temperatura = povprečje zraka)
zap = np.zeros((len(T_zrak), len(z)))
ure = np.arange(len(T_zrak)) / 24
fig = plt.figure(figsize=(12, 4))
ax = fig.add_subplot(121)
for zi in (0, 0.1, 0.2, 0.5):
    ax.plot(ure[4344:4344 + 168], zap[4344:4344 + 168, int(round(zi / dz))], label=f"$z = {zi}$ m")
ax.set_xlabel("dan v letu")
ax.set_ylabel("$T$ [°C]")
ax.set_title("teden v juliju", loc="left")
ax.legend()
ax = fig.add_subplot(122)
for zi in (0, 1, 2, 4):
    ax.plot(ure, zap[:, int(round(zi / dz))], lw=0.8, label=f"$z = {zi}$ m")
ax.set_xlabel("dan v letu")
ax.set_ylabel("$T$ [°C]")
ax.set_title("celo leto", loc="left")
ax.legend()
fig.tight_layout()
plt.show()

# Naloga 2 - Slika 4

In [ ]:
globine = np.array([0, 0.1, 0.2, 0.5, 1, 2, 4])
t_ure = np.arange(len(T_zrak)) + 1.0                 # profil je zapisan ob koncu ure
# TODO: dnevna (24 h) in letna (8760 h) amplituda ter zamik za vse globine (Preglednica 2); delta iz naklona ln A(z)
A_d = A_l = zam_d = zam_l = np.ones(len(globine))
fig = plt.figure(figsize=(12, 4))
ax = fig.add_subplot(121)
ax.semilogy(globine[:4], A_d[:4], "o-", label="dnevna")
ax.semilogy(globine, A_l, "s-", label="letna")
ax.set_xlabel("$z$ [m]")
ax.set_ylabel("amplituda [°C]")
ax.legend()
ax = fig.add_subplot(122)
ax.plot(globine[:4], zam_d[:4], "o-", label="dnevna [h]")
ax.plot(globine, zam_l, "s-", label="letna [dni]")
ax.set_xlabel("$z$ [m]")
ax.set_ylabel("zamik")
ax.legend()
fig.tight_layout()
plt.show()

# Naloga 3 - Slika 5

In [ ]:
T_hlad, T_voda, N_r = 5.0, 100.0, 44
koef = {0: 4 / np.pi, 1: 2 / (beta[1] * j1(beta[1])), 2: 2.0}   # prvi člen vrste v središču
fig = plt.figure(figsize=(12, 4))
ax1, ax2 = fig.add_subplot(121), fig.add_subplot(122)
# TODO: za d = 0, 1, 2 temperatura v središču, tau iz naklona ln(theta), čas do 70 °C (Preglednica 3)
pass
ax1.axhline(70, color="k", ls=":")
ax1.set_xlabel("$t$ [min]")
ax1.set_ylabel("$T$ v središču [°C]")
ax1.legend()
ax2.set_xlabel("$t$ [min]")
ax2.set_ylabel(r"$\theta = (T_v - T_c)/(T_v - T_0)$")
ax2.set_ylim(1e-3, 1.5)
ax2.legend()
fig.tight_layout()
plt.show()

# Naloga 4 - Slika 6

In [ ]:
a_v, d_st, h_v = 1.5, 0.30, 0.01
h_i, h_e, T_i, T_e = 1 / 0.13, 1 / 0.04, 20.0, -5.0
tip = vogal(a_v, d_st, h_v)
n_v = tip.shape[0] - 2
xc = (np.arange(n_v) + 0.5) * h_v
X, Y = np.meshgrid(xc, xc)
zid = tip[1:-1, 1:-1] == ZID
L_n = 2 * (a_v - d_st)                                 # notranja dolžina obeh krakov
rezultati = {}
# TODO: za lam = 0.8 in 0.2: U ravne stene, stacionarna T, tok Q, psi, najnižja T_si in f_Rsi (Preglednica 4)
pass
T = rezultati[0.8][0]
nd = int(round(d_st / h_v))
fig = plt.figure(figsize=(12, 5))
ax = fig.add_subplot(121)
Tz = np.where(zid, T[1:-1, 1:-1], np.nan)
cs = ax.contourf(X, Y, Tz, levels=np.arange(-5, 21, 1), cmap="coolwarm")
ax.contour(X, Y, Tz, levels=np.arange(-4, 20, 2), colors="k", linewidths=0.5)
fig.colorbar(cs, ax=ax, label="$T$ [°C]")
ax.set_xlim(0, 1.0)
ax.set_ylim(0, 1.0)
ax.set_aspect("equal")
ax.set_xlabel("$x$ [m]")
ax.set_ylabel("$y$ [m]")
ax.set_title(r"$\lambda = 0{,}8$ W/(m K)", loc="left")
ax = fig.add_subplot(122)
for lam, (T_, T_si) in rezultati.items():
    ax.plot(xc[nd:] - d_st, T_si[nd:, nd - 1], label=rf"$\lambda = {lam}$ W/(m K)")   # notranja površina navpičnega kraka
ax.axhline(T_e + 0.7 * (T_i - T_e), color="k", ls=":", label="$f_{Rsi} = 0{,}7$")
ax.set_xlabel("razdalja od notranjega vogala [m]")
ax.set_ylabel("$T_{si}$ [°C]")
ax.set_title("temperatura notranje površine", loc="left")
ax.legend()
fig.tight_layout()
plt.show()